# TODO

- nearest neighbor query
- model selection (1, 5, 10, 15, 20 neighbors)

In [1]:
import numpy as np
import pandas as pd
import pickle

from pathlib import Path
from itertools import product
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from numpy.lib.stride_tricks import sliding_window_view

from aipad.pad_imputation import (
    make_train_test, form_test_matrices,
    plot_results, read_npzs, save_results
)
from aipad.spacecrafts import SoloConstants, WindConstants

solo = SoloConstants()
wind = WindConstants()

In [2]:
bin_width_deg = 1
time_avg_min = 1
avg_bin_dir = f"{time_avg_min}min_{bin_width_deg}deg"

load_path = Path("./data/intensities") / avg_bin_dir
cov_path = Path("./data/coverages") / avg_bin_dir
plot_path = Path(f"./plots/")
model_path = Path("./data/models")

plot_path.mkdir(exist_ok=True)
model_path.mkdir(exist_ok=True)

In [3]:
# Train test split: pick e.g. every 3rd as test (train, train, test, train, train, test...)
# Events are in time order so this takes the solar cycle into account.
# NOTE: using future observations to predict past observations is generally not OK, but
# the nature of the data is such that this can be ignored.

hist_arr, reduced_hist_arr, intensity_arr, metadata_arr = read_npzs(load_path=load_path)

X_train, X_test, y_train, y_test, I_train, I_test, meta_train, meta_test = make_train_test(hist_arr, reduced_hist_arr, intensity_arr, metadata_arr, n=3)

# Stacking of training cases for KNN imputer:
# concatenate 5 successive rows to one feature vector (5 * 180 = 900 features)
trains = []
for train in X_train:
    train_stacked = sliding_window_view(train, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    trains.append(train_stacked)

X_train_stacked = np.vstack(trains)

### Mean value imputation
Two ways to do this with either rows as samples or columns as samples. `SimpleImputer()` uses mean of column as the value to fill.

TODO/Ideas:
- Dividing into smaller chunks probably makes it better (in both cases), since then the variations over large scales don't affect the local mean

In [ ]:
# fitting to test cases since "fitting" is just calculating the means of each column. "Supervised learning" approach is not applicable here
# Columns as samples: each time bin is filled with the mean across the whole angle space.
for j in range(0, len(y_test)):
    model = SimpleImputer(keep_empty_features=True) 
    res = form_test_matrices(model, X_test[j], y_test[j], transpose=True)
    plot_results(model, res, I_test[j], sc=wind, cov_sc=solo, save_path=Path("./plots") / "meanimputer" / f"results_{j}_columns-as-samples.png")

/home/ojsant/miniconda3/envs/aipad/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ojsant/miniconda3/envs/aipad/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ojsant/miniconda3/envs/aipad/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ojsant/miniconda3/envs/aipad/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/ojsant/miniconda3/envs/aipad/lib/python3.13/site-packages/sklearn/metrics/_regress

### KNNImputer

In [ ]:
neighbors = 5

model = KNNImputer(n_neighbors=neighbors, weights="distance", keep_empty_features=True)
model.fit(X_train_stacked)
dump_file = open(model_path / f'knnimputer_20250111', 'wb')
pickle.dump(model, dump_file)
dump_file.close()

In [ ]:
for j in range(0, len(y_test)):
    res = form_test_matrices(model, X_test[j], y_test[j])
    plot_results(model, res, "mse", I_test[j], sc=wind, cov_sc=solo, save_path=plot_path / f"results_{j}.png")

In [ ]:
# Model selection CV
neighbors = [1, 5, 10, 15]
weights = ["distance", "uniform"]
n_splits = 3
n_tests = 5
# Cross validation: shift train-test split by one 3 times (a sort of 3-fold CV),
# but test only 5 first events in each fold
for shift in range(n_splits):
    print(f"Split {shift}")
    result_df = pd.DataFrame()
    (X_train, X_test, y_train, y_test,
     I_train, I_test, meta_train, meta_test) = make_train_test(hist_arr, reduced_hist_arr,
                                                               intensity_arr, metadata_arr,
                                                               n=n_splits, shift=shift)
    trains = []
    for train in X_train:
        train_stacked = sliding_window_view(train, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
        trains.append(train_stacked)

    X_train_stacked = np.vstack(trains)
    
    for n, w in product(neighbors, weights):
        model = KNNImputer(n_neighbors=n, weights=w, keep_empty_features=True)
        model.fit(X_train_stacked)
        
        for j in range(n_tests):
            res = form_test_matrices(model, X_test[j], y_test[j])
            plot_results(model, res, "mse", I_test[j], sc=wind, cov_sc=solo,
                         save_path=plot_path / "knnimputer" / "crossvalidation" / f"split_{shift}_results_{j}.png")
            result_df = save_results(model, res, meta_test[j], "mse", Path(f"./knnimputer_results_{shift}.csv"))
            print(f"Test {j} complete")

        print(f"Mean score on fold {}")
    print(f"Split {shift} results:\n")
    print(result_df[["event_df", "score", "model_params"]])
    print("Mean (CV) score:", result_df.score.mean())

Split 0
Fold 0 complete
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 0 complete
Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete


KeyboardInterrupt: 

In [5]:
pd.read_csv("knnimputer_results_0.csv")

,event_dt,miss_percent,score,scorer,model_params
0,2012-07-04 17:07:36,46.075617,7192.317330,mse,"(1, 'uniform')"
1,2010-08-24 09:00:00,47.560185,941.844507,mse,"(1, 'uniform')"
2,2012-10-31 14:42:37,51.504630,446.832114,mse,"(1, 'uniform')"
3,2012-10-07 11:00:00,49.222994,312996.878971,mse,"(1, 'uniform')"
4,2010-08-18 05:52:40,52.053241,136126.780583,mse,"(1, 'uniform')"
5,2012-07-04 17:07:36,46.075617,7192.317330,mse,"(1, 'distance')"
6,2010-08-24 09:00:00,47.560185,941.844507,mse,"(1, 'distance')"
7,2012-10-31 14:42:37,51.504630,446.832114,mse,"(1, 'distance')"
8,2012-10-07 11:00:00,49.222994,312996.878971,mse,"(1, 'distance')"
9,2010-08-18 05:52:40,52.053241,136126.780583,mse,"(1, 'distance')"


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_features=32)

# X = true, y = reduced. Train model with y as input, X as output
X_trains = []
y_trains = []
for X, y in zip(X_train, y_train):
    X_stacked = sliding_window_view(X, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])
    y_stacked = sliding_window_view(y, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    y_trains.append(y_stacked)
    X_trains.append(X_stacked)

X_train_stacked = np.vstack(X_trains)
y_train_stacked = np.vstack(y_trains)

In [1]:
from sklearn.metrics import nan_euclidean_distances
from sklearn.impute import KNNImputer